# Task 3.3 — Gold: Course Completion Funnel
## Notebook: 10_gold_completion_funnel

**Pipeline:** EduPath Learning Analytics · Medallion Architecture
**Layer:** Gold (funnel analytics — drives curriculum improvement)

**What this notebook does:**
- Computes a 6-stage completion funnel per course_id:
    Stage 1: enrolled        → all students who enrolled
    Stage 2: started         → had at least one login event AFTER enrollment date
    Stage 3: reached_25pct   → current_progress_pct >= 25
    Stage 4: reached_50pct   → current_progress_pct >= 50
    Stage 5: reached_75pct   → current_progress_pct >= 75
    Stage 6: completed       → enrollment status = COMPLETED
- Calculates % of enrolled students who reached each milestone
- Identifies biggest_dropoff_stage: the stage with the largest absolute drop
- Cohort analysis: Q1 (Jan–Mar) vs Q2 (Apr–Jun) enrolled students
  comparison of completed_pct per course
- Writes to gold.course_completion_funnel

**Source tables:**
    mini_project_grp6.bronze.enrollments_bronze
    mini_project_grp6.bronze.lms_events_bronze

**Target table:** mini_project_grp6.gold.course_completion_funnel

**Business Rules enforced:**
- enrolled_count must be >= 1 for every row
- All pct columns must be between 0.0 and 1.0
- started_pct >= completed_pct (you cannot complete without starting)
- biggest_dropoff_stage must be one of the 5 valid transition labels
- cohort_q1_completion and cohort_q2_completion must be between 0.0 and 1.0

**Run order:** Run all cells top-to-bottom. Safe to re-run (OVERWRITE mode).

In [0]:
# ============================================================
# CELL 2 — Catalog config and all path/table name constants
# Run this cell first every time before executing any other cell
# ============================================================

# --- Catalog & schema names ---
CATALOG = "mini_project_grp6"
BRONZE  = "bronze"
GOLD    = "gold"

# --- Source tables ---
ENROLL_BRONZE_TABLE = f"{CATALOG}.{BRONZE}.enrollments_bronze"
LMS_BRONZE_TABLE    = f"{CATALOG}.{BRONZE}.lms_events_bronze"

# --- Target table ---
FUNNEL_GOLD_TABLE   = f"{CATALOG}.{GOLD}.course_completion_funnel"
DQ_AUDIT_TABLE      = f"{CATALOG}.audit.dq_audit"

# --- Cohort date boundaries ---
# Q1: January 1 → March 31
# Q2: April 1   → June 30
Q1_START = "2024-01-01"
Q1_END   = "2024-03-31"
Q2_START = "2024-04-01"
Q2_END   = "2024-06-30"

# --- Progress milestone thresholds ---
PCT_25 = 25.0
PCT_50 = 50.0
PCT_75 = 75.0

# --- Schema version tag ---
SCHEMA_VERSION = "v1.0"

# --- Funnel stage transition names (used in biggest_dropoff_stage) ---
STAGE_TRANSITIONS = [
    "enrolled→started",
    "started→25pct",
    "25pct→50pct",
    "50pct→75pct",
    "75pct→completed"
]

# --- Set default catalog + schema ---
spark.catalog.setCurrentCatalog(CATALOG)
spark.catalog.setCurrentDatabase(GOLD)

print(f"✅ Catalog : {CATALOG}")
print(f"✅ Schema  : {GOLD}")
print(f"✅ Target  : {FUNNEL_GOLD_TABLE}")
print()
print(f"Cohort definitions:")
print(f"   Q1: {Q1_START} → {Q1_END}")
print(f"   Q2: {Q2_START} → {Q2_END}")
print()
print(f"Funnel stages: enrolled → started → 25% → 50% → 75% → completed")

**Expected output:**
```
✅ Catalog : mini_project_grp6
✅ Schema  : gold
✅ Target  : mini_project_grp6.gold.course_completion_funnel

Cohort definitions:
   Q1: 2024-01-01 → 2024-03-31
   Q2: 2024-04-01 → 2024-06-30

Funnel stages: enrolled → started → 25% → 50% → 75% → completed
```

In [0]:
# ============================================================
# CELL 3 — All imports needed for this notebook
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import (
    StringType, IntegerType,
    DoubleType, DateType
)
from delta.tables import DeltaTable

print("✅ Imports successful")

## Part A — Read Source Tables + Validation

Two sources feed this notebook:

1. enrollments_bronze  — progress_pct, status, enrolled_date, last_accessed_date
   This gives us stages 1, 3, 4, 5, 6 directly.

2. lms_events_bronze   — event_type = 'login', event_date
   Used to determine Stage 2 (started = had a login AFTER enrollment date).
   We join login events back to enrollments on student_id + course_id.

In [0]:
# ============================================================
# CELL 5 — Read both source tables and validate
# ============================================================

enroll_df = spark.table(ENROLL_BRONZE_TABLE)
lms_df    = spark.table(LMS_BRONZE_TABLE)

enroll_total = enroll_df.count()
lms_total    = lms_df.count()

print("── Source validation ─────────────────────────────")
print(f"   enrollments_bronze : {enroll_total:,} rows")
print(f"   lms_events_bronze  : {lms_total:,} rows")
print()

print("── enrollment status distribution ────────────────")
enroll_df.groupBy("status").count().orderBy("status").show()

print("── progress_pct stats ────────────────────────────")
enroll_df.select(
    F.min("current_progress_pct").alias("min_pct"),
    F.max("current_progress_pct").alias("max_pct"),
    F.round(F.avg("current_progress_pct"), 2).alias("avg_pct")
).show()

print("── enrolled_date range ───────────────────────────")
enroll_df.select(
    F.min("enrolled_date").alias("earliest_enrollment"),
    F.max("enrolled_date").alias("latest_enrollment")
).show()

# Filter to only courses and students with valid IDs before funnel computation
enroll_clean_df = enroll_df.filter(
    F.col("student_id").isNotNull() &
    F.col("course_id").isNotNull() &
    F.col("enrolled_date").isNotNull()
)

clean_count = enroll_clean_df.count()
print(f"ℹ  Rows after NULL filter: {clean_count:,}")

## Part B — Identify "Started" Students (Stage 2)

Stage 2 definition: a student has "started" a course if they
had at least one login event AFTER their enrolled_date for that course.

Approach:
1. Get all login events from lms_events_bronze
2. Join to enrollments on (student_id, course_id)
3. Filter: event_date >= enrolled_date (login must be on or after enrollment)
4. Any student-course pair that appears in this join has "started"

We produce a set of (student_id, course_id) pairs that represent
students who have started — this becomes a boolean flag in the funnel.

In [0]:
# ============================================================
# CELL 7 — Identify students who "started" each course
#
# Started = at least one login event on or after their enrolled_date
# for the same course_id.
#
# We use a semi-join approach:
#   1. Filter LMS events to login events only
#   2. Join to enrollments on (student_id, course_id)
#   3. Keep only rows where event_date >= enrolled_date
#   4. Distinct (student_id, course_id) pairs = "started" set
# ============================================================

# Step 1: get all login events
login_events_df = (
    lms_df
    .filter(
        (F.col("event_type") == "login") &
        F.col("student_id").isNotNull() &
        F.col("course_id").isNotNull() &
        F.col("event_date").isNotNull()
    )
    .select("student_id", "course_id", "event_date")
)

# Step 2 & 3: join to enrollments + filter to logins after enrollment
started_df = (
    enroll_clean_df
    .select("student_id", "course_id", "enrolled_date")
    .join(login_events_df, on=["student_id", "course_id"], how="inner")
    .filter(F.col("event_date") >= F.col("enrolled_date"))
    # Step 4: distinct student-course pairs who started
    .select("student_id", "course_id")
    .distinct()
    .withColumn("has_started", F.lit(True))
)

started_count = started_df.count()
print("── Started students identification ───────────────")
print(f"   (student_id, course_id) pairs who started: {started_count:,}")
print(f"   Total enrollments                         : {enroll_clean_df.count():,}")
print(f"   Start rate                                : {started_count / enroll_clean_df.count():.2%}")

## Part C — Build Per-Student Funnel Stage Flags

For each (student_id, course_id) enrollment, we create 6 boolean columns:

  is_enrolled    : always True (every row in enrollments)
  is_started     : joins from started_df (had login after enrollment)
  is_25pct       : current_progress_pct >= 25
  is_50pct       : current_progress_pct >= 50
  is_75pct       : current_progress_pct >= 75
  is_completed   : enrollment status = 'COMPLETED'

These boolean flags are then aggregated per course_id to compute
the funnel percentages.

In [0]:
# ============================================================
# CELL 9 — Add boolean stage flags to each enrollment
#
# Join started_df onto enrollments (left join so non-starters
# get has_started = False via fillna).
# Then derive the remaining flags from progress_pct and status.
# ============================================================

enrollment_with_flags_df = (
    enroll_clean_df
    # Join started flag
    .join(started_df, on=["student_id", "course_id"], how="left")
    .fillna({"has_started": False})
    # Add milestone flags
    .withColumn("is_enrolled",  F.lit(True))
    .withColumn("is_started",   F.col("has_started"))
    .withColumn("is_25pct",
        F.col("current_progress_pct") >= F.lit(PCT_25)
    )
    .withColumn("is_50pct",
        F.col("current_progress_pct") >= F.lit(PCT_50)
    )
    .withColumn("is_75pct",
        F.col("current_progress_pct") >= F.lit(PCT_75)
    )
    .withColumn("is_completed",
        F.col("status") == "COMPLETED"
    )
    # Keep enrolled_date for cohort analysis
    .select(
        "student_id",
        "course_id",
        "enrolled_date",
        "is_enrolled",
        "is_started",
        "is_25pct",
        "is_50pct",
        "is_75pct",
        "is_completed",
    )
)

total_with_flags = enrollment_with_flags_df.count()
print("── Stage flags added ─────────────────────────────")
print(f"   Total enrollment rows: {total_with_flags:,}")
print()

print("── Stage flag totals across all courses ──────────")
enrollment_with_flags_df.select(
    F.sum(F.col("is_enrolled").cast("int")).alias("enrolled"),
    F.sum(F.col("is_started").cast("int")).alias("started"),
    F.sum(F.col("is_25pct").cast("int")).alias("at_25pct"),
    F.sum(F.col("is_50pct").cast("int")).alias("at_50pct"),
    F.sum(F.col("is_75pct").cast("int")).alias("at_75pct"),
    F.sum(F.col("is_completed").cast("int")).alias("completed")
).show()

## Part D — Aggregate Funnel Per course_id (All Students)

Group by course_id and count how many students reached each stage.
Then compute pct columns:

  started_pct   = started_count / enrolled_count
  q1_pct        = at_25pct_count / enrolled_count
  q2_pct        = at_50pct_count / enrolled_count
  q3_pct        = at_75pct_count / enrolled_count
  completed_pct = completed_count / enrolled_count

Note: "q1_pct", "q2_pct", "q3_pct" in the column names refer to
progress quartiles (25%, 50%, 75%) — NOT calendar Q1/Q2.
The architecture document uses this naming convention.
Calendar cohort analysis (Q1 Jan-Mar vs Q2 Apr-Jun) is separate.

In [0]:
# ============================================================
# CELL 11 — Aggregate funnel per course_id
#
# enrolled_count : total distinct students enrolled
# started_count  : students with at least one post-enrollment login
# count_25pct    : students with progress >= 25%
# count_50pct    : students with progress >= 50%
# count_75pct    : students with progress >= 75%
# completed_count: students with status = COMPLETED
#
# pct columns = count / enrolled_count, rounded to 4 decimal places
# ============================================================

funnel_counts_df = (
    enrollment_with_flags_df
    .groupBy("course_id")
    .agg(
        # Count at each stage
        F.sum(F.col("is_enrolled").cast("int")).alias("enrolled_count"),
        F.sum(F.col("is_started").cast("int")).alias("started_count"),
        F.sum(F.col("is_25pct").cast("int")).alias("count_25pct"),
        F.sum(F.col("is_50pct").cast("int")).alias("count_50pct"),
        F.sum(F.col("is_75pct").cast("int")).alias("count_75pct"),
        F.sum(F.col("is_completed").cast("int")).alias("completed_count"),
    )
)

# Compute pct columns — enrolled_count is always >= 1 (we have enrollments)
funnel_pct_df = (
    funnel_counts_df
    .withColumn(
        "started_pct",
        F.round(F.col("started_count") / F.col("enrolled_count"), 4)
    )
    .withColumn(
        "q1_pct",
        F.round(F.col("count_25pct") / F.col("enrolled_count"), 4)
    )
    .withColumn(
        "q2_pct",
        F.round(F.col("count_50pct") / F.col("enrolled_count"), 4)
    )
    .withColumn(
        "q3_pct",
        F.round(F.col("count_75pct") / F.col("enrolled_count"), 4)
    )
    .withColumn(
        "completed_pct",
        F.round(F.col("completed_count") / F.col("enrolled_count"), 4)
    )
)

print("── Funnel aggregation per course ─────────────────")
print(f"   Distinct courses: {funnel_pct_df.count():,}")
print()

print("── Overall funnel (avg % across all courses) ─────")
funnel_pct_df.select(
    F.round(F.avg("started_pct"), 4).alias("avg_started_pct"),
    F.round(F.avg("q1_pct"), 4).alias("avg_q1_pct"),
    F.round(F.avg("q2_pct"), 4).alias("avg_q2_pct"),
    F.round(F.avg("q3_pct"), 4).alias("avg_q3_pct"),
    F.round(F.avg("completed_pct"), 4).alias("avg_completed_pct")
).show()

## Part E — Identify biggest_dropoff_stage per Course

For each course, find the funnel transition where the largest
absolute percentage of students dropped off.

The 5 transitions are:
  1. enrolled→started   : started_pct - 1.0          (drop from 100% enrolled)
  2. started→25pct      : q1_pct - started_pct
  3. 25pct→50pct        : q2_pct - q1_pct
  4. 50pct→75pct        : q3_pct - q2_pct
  5. 75pct→completed    : completed_pct - q3_pct

Each drop value is negative (students fall off).
biggest_dropoff_stage = the transition with the most negative drop value
(i.e., where the most students stopped progressing).

In [0]:
# ============================================================
# CELL 13 — Compute biggest_dropoff_stage
#
# For each of the 5 transitions, compute the drop in pct.
# Drop = next_stage_pct - current_stage_pct (negative = students lost)
#
# Use a named struct approach:
#   Create a column for each transition's drop value
#   Use F.least() to find the minimum (most negative drop)
#   Then map that back to the transition name using F.when() chains
# ============================================================

funnel_with_drops_df = (
    funnel_pct_df
    # Compute drop at each transition
    .withColumn(
        "drop_enrolled_started",
        F.round(F.col("started_pct") - F.lit(1.0), 4)
    )
    .withColumn(
        "drop_started_25",
        F.round(F.col("q1_pct") - F.col("started_pct"), 4)
    )
    .withColumn(
        "drop_25_50",
        F.round(F.col("q2_pct") - F.col("q1_pct"), 4)
    )
    .withColumn(
        "drop_50_75",
        F.round(F.col("q3_pct") - F.col("q2_pct"), 4)
    )
    .withColumn(
        "drop_75_completed",
        F.round(F.col("completed_pct") - F.col("q3_pct"), 4)
    )
)

# Find the minimum drop value (most negative = biggest loss) per row
funnel_with_min_drop_df = (
    funnel_with_drops_df
    .withColumn(
        "min_drop",
        F.least(
            F.col("drop_enrolled_started"),
            F.col("drop_started_25"),
            F.col("drop_25_50"),
            F.col("drop_50_75"),
            F.col("drop_75_completed")
        )
    )
    # Map the min_drop back to the transition label
    .withColumn(
        "biggest_dropoff_stage",
        F.when(
            F.col("min_drop") == F.col("drop_enrolled_started"),
            F.lit("enrolled→started")
        ).when(
            F.col("min_drop") == F.col("drop_started_25"),
            F.lit("started→25pct")
        ).when(
            F.col("min_drop") == F.col("drop_25_50"),
            F.lit("25pct→50pct")
        ).when(
            F.col("min_drop") == F.col("drop_50_75"),
            F.lit("50pct→75pct")
        ).otherwise(
            F.lit("75pct→completed")
        )
    )
    # Drop intermediate drop columns — keep only the label and min_drop value
    .drop(
        "drop_enrolled_started", "drop_started_25",
        "drop_25_50", "drop_50_75", "drop_75_completed",
        "started_count", "count_25pct", "count_50pct",
        "count_75pct", "completed_count"
    )
)

print("── biggest_dropoff_stage distribution ────────────")
funnel_with_min_drop_df.groupBy("biggest_dropoff_stage") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(truncate=False)

print("── Sample: courses with worst completion funnel ──")
funnel_with_min_drop_df.orderBy(F.asc("completed_pct")).select(
    "course_id", "enrolled_count", "started_pct",
    "q1_pct", "q2_pct", "q3_pct", "completed_pct",
    "biggest_dropoff_stage", "min_drop"
).show(5, truncate=False)

## Part F — Q1 vs Q2 Cohort Analysis

Cohort analysis compares completion rates for:
- Q1 cohort: students whose enrolled_date is Jan 1 – Mar 31
- Q2 cohort: students whose enrolled_date is Apr 1 – Jun 30

Per course_id we compute:
- cohort_q1_enrolled    : count of Q1 students enrolled
- cohort_q1_completion  : completed_count / enrolled_count for Q1 students
- cohort_q2_enrolled    : count of Q2 students enrolled
- cohort_q2_completion  : completed_count / enrolled_count for Q2 students

These feed curriculum improvement decisions:
"Did the course updates made between Q1 and Q2 improve completion?"

In [0]:
# ============================================================
# CELL 15 — Q1 cohort: students enrolled Jan 1 – Mar 31
#
# For each course, count Q1 students and their completion rate.
# Uses enrollment_with_flags_df which already has stage flags.
# ============================================================

q1_cohort_df = (
    enrollment_with_flags_df
    .filter(
        (F.col("enrolled_date") >= F.lit(Q1_START)) &
        (F.col("enrolled_date") <= F.lit(Q1_END))
    )
    .groupBy("course_id")
    .agg(
        F.sum(F.col("is_enrolled").cast("int")).alias("cohort_q1_enrolled"),
        F.sum(F.col("is_completed").cast("int")).alias("cohort_q1_completed"),
    )
    .withColumn(
        "cohort_q1_completion",
        F.round(
            F.when(
                F.col("cohort_q1_enrolled") > 0,
                F.col("cohort_q1_completed") / F.col("cohort_q1_enrolled")
            ).otherwise(F.lit(None).cast(DoubleType())),
            4
        )
    )
    .drop("cohort_q1_completed")
)

q1_course_count = q1_cohort_df.count()
print("── Q1 Cohort ─────────────────────────────────────")
print(f"   Courses with Q1 enrollments: {q1_course_count:,}")
print()
q1_cohort_df.select(
    F.sum("cohort_q1_enrolled").alias("total_q1_students"),
    F.round(F.avg("cohort_q1_completion"), 4).alias("avg_q1_completion_rate")
).show()

In [0]:
# ============================================================
# CELL 16 — Q2 cohort: students enrolled Apr 1 – Jun 30
# Same logic as Q1 but filtered to Q2 date range
# ============================================================

q2_cohort_df = (
    enrollment_with_flags_df
    .filter(
        (F.col("enrolled_date") >= F.lit(Q2_START)) &
        (F.col("enrolled_date") <= F.lit(Q2_END))
    )
    .groupBy("course_id")
    .agg(
        F.sum(F.col("is_enrolled").cast("int")).alias("cohort_q2_enrolled"),
        F.sum(F.col("is_completed").cast("int")).alias("cohort_q2_completed"),
    )
    .withColumn(
        "cohort_q2_completion",
        F.round(
            F.when(
                F.col("cohort_q2_enrolled") > 0,
                F.col("cohort_q2_completed") / F.col("cohort_q2_enrolled")
            ).otherwise(F.lit(None).cast(DoubleType())),
            4
        )
    )
    .drop("cohort_q2_completed")
)

q2_course_count = q2_cohort_df.count()
print("── Q2 Cohort ─────────────────────────────────────")
print(f"   Courses with Q2 enrollments: {q2_course_count:,}")
print()
q2_cohort_df.select(
    F.sum("cohort_q2_enrolled").alias("total_q2_students"),
    F.round(F.avg("cohort_q2_completion"), 4).alias("avg_q2_completion_rate")
).show()

## Part G — Join Cohort Metrics + Compute cohort_improvement_flag

Join Q1 and Q2 cohort metrics onto the funnel DataFrame.
For courses that appear in both cohorts, compute:

  cohort_q2_vs_q1_delta = cohort_q2_completion - cohort_q1_completion
  cohort_improvement    = "IMPROVED"  if delta > 0.02
                        = "DECLINED"  if delta < -0.02
                        = "STABLE"    otherwise
                        = NULL        if course has only one cohort

This tells curriculum designers whether Q2 students are completing
better or worse than Q1 students — a direct signal of course effectiveness.

In [0]:
# ============================================================
# CELL 18 — Join cohort metrics + compute cohort comparison
#           Assemble final Gold DataFrame with all columns
#           Add Gold metadata columns
# ============================================================

# Join Q1 and Q2 cohorts onto the funnel DataFrame (left joins)
funnel_with_cohorts_df = (
    funnel_with_min_drop_df
    .join(q1_cohort_df, on="course_id", how="left")
    .join(q2_cohort_df, on="course_id", how="left")
    # Cohort delta: positive = Q2 students completing better than Q1
    .withColumn(
        "cohort_q2_vs_q1_delta",
        F.when(
            F.col("cohort_q1_completion").isNotNull() &
            F.col("cohort_q2_completion").isNotNull(),
            F.round(
                F.col("cohort_q2_completion") - F.col("cohort_q1_completion"),
                4
            )
        ).otherwise(F.lit(None).cast(DoubleType()))
    )
    # Cohort improvement label
    .withColumn(
        "cohort_improvement",
        F.when(F.col("cohort_q2_vs_q1_delta").isNull(), F.lit(None).cast(StringType()))
         .when(F.col("cohort_q2_vs_q1_delta") > F.lit(0.02),  F.lit("IMPROVED"))
         .when(F.col("cohort_q2_vs_q1_delta") < F.lit(-0.02), F.lit("DECLINED"))
         .otherwise(F.lit("STABLE"))
    )
)

# Assemble final column order with metadata
completion_funnel_df = (
    funnel_with_cohorts_df
    .select(
        # ── Grain ─────────────────────────────────────────────
        "course_id",
        # ── Funnel counts ─────────────────────────────────────
        "enrolled_count",
        # ── Funnel percentages ────────────────────────────────
        "started_pct",
        "q1_pct",
        "q2_pct",
        "q3_pct",
        "completed_pct",
        # ── Drop-off analysis ─────────────────────────────────
        "biggest_dropoff_stage",
        "min_drop",
        # ── Cohort analysis ───────────────────────────────────
        "cohort_q1_enrolled",
        "cohort_q1_completion",
        "cohort_q2_enrolled",
        "cohort_q2_completion",
        "cohort_q2_vs_q1_delta",
        "cohort_improvement",
    )
    # Add Gold metadata
    .withColumn("_gold_load_ts",   F.current_timestamp())
    .withColumn("_schema_version", F.lit(SCHEMA_VERSION))
)

print("── Final Completion Funnel DataFrame ─────────────")
print(f"   Rows    : {completion_funnel_df.count():,}")
print(f"   Columns : {len(completion_funnel_df.columns)}")
print()
completion_funnel_df.printSchema()

In [0]:
# ============================================================
# CELL 19 — Write to gold.course_completion_funnel
#           Mode: OVERWRITE — one row per course, fully recomputed
#           No partitioning needed (course_id is the grain,
#           not a time-series table — one snapshot of the funnel)
# ============================================================

(
    completion_funnel_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(FUNNEL_GOLD_TABLE)
)

final_count = spark.table(FUNNEL_GOLD_TABLE).count()
print(f"✅ Written to: {FUNNEL_GOLD_TABLE}")
print(f"   Rows in table (one per course): {final_count:,}")

**Expected output:**
```
✅ Written to: mini_project_grp6.gold.course_completion_funnel
   Rows in table (one per course): 250
```

In [0]:
# ============================================================
# CELL 20 — Verify gold.course_completion_funnel
# ============================================================

funnel_df = spark.table(FUNNEL_GOLD_TABLE)

print("── course_completion_funnel ──────────────────────")
print(f"Total rows   : {funnel_df.count():,}")
print(f"Columns      : {len(funnel_df.columns)}")
print()

print("── Overall funnel (avg % across all courses) ─────")
funnel_df.select(
    F.round(F.avg("started_pct"), 4).alias("avg_started_pct"),
    F.round(F.avg("q1_pct"), 4).alias("avg_q1_pct"),
    F.round(F.avg("q2_pct"), 4).alias("avg_q2_pct"),
    F.round(F.avg("q3_pct"), 4).alias("avg_q3_pct"),
    F.round(F.avg("completed_pct"), 4).alias("avg_completed_pct")
).show()

print("── biggest_dropoff_stage distribution ────────────")
funnel_df.groupBy("biggest_dropoff_stage") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(truncate=False)

print("── cohort_improvement distribution ──────────────")
funnel_df.groupBy("cohort_improvement") \
    .count() \
    .show(truncate=False)

print("── Q1 vs Q2 cohort comparison ────────────────────")
funnel_df.select(
    F.round(F.avg("cohort_q1_completion"), 4).alias("avg_q1_completion"),
    F.round(F.avg("cohort_q2_completion"), 4).alias("avg_q2_completion"),
    F.round(F.avg("cohort_q2_vs_q1_delta"), 4).alias("avg_q2_vs_q1_delta")
).show()

print("── Top 5 courses with biggest Q2 improvement ─────")
funnel_df.filter(F.col("cohort_improvement") == "IMPROVED") \
    .orderBy(F.desc("cohort_q2_vs_q1_delta")) \
    .select(
        "course_id", "cohort_q1_completion",
        "cohort_q2_completion", "cohort_q2_vs_q1_delta",
        "completed_pct", "biggest_dropoff_stage"
    ).show(5, truncate=False)

## Part H — Data Quality Checks (Gold)

DQ checks for course_completion_funnel:
1. enrolled_count must be >= 1 for every row
2. All pct columns must be between 0.0 and 1.0
3. Funnel must be monotonically non-increasing:
   started_pct >= q1_pct >= q2_pct >= q3_pct >= completed_pct
   (you cannot reach 50% without passing 25%)
4. biggest_dropoff_stage must be one of the 5 valid transition labels
5. cohort_q1_completion and cohort_q2_completion must be in [0.0, 1.0]
   when not NULL

In [0]:
# ============================================================
# CELL 22 — DQ Check 1: enrolled_count must be >= 1
#           A 0-enrolled course row indicates a join error
# ============================================================

funnel_df = spark.table(FUNNEL_GOLD_TABLE)

zero_enrolled = (
    funnel_df
    .filter(F.col("enrolled_count") < 1)
    .withColumn("dq_check_name", F.lit("enrolled_count_zero"))
    .withColumn("dq_table",      F.lit(FUNNEL_GOLD_TABLE))
    .withColumn("dq_severity",   F.lit("HIGH"))
    .withColumn("dq_checked_at", F.current_timestamp())
    .withColumn("dq_message",    F.concat_ws(" | ",
        F.lit("enrolled_count < 1 for course_id:"),
        F.col("course_id")
    ))
)

zero_count = zero_enrolled.count()
print("── DQ Check 1: enrolled_count >= 1 ──────────────")
if zero_count > 0:
    (
        zero_enrolled
        .select("dq_check_name", "dq_table", "dq_severity",
                "dq_checked_at", "dq_message", "course_id")
        .write.format("delta").mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(DQ_AUDIT_TABLE)
    )
    print(f"   ⚠ {zero_count} rows with enrolled_count < 1 → {DQ_AUDIT_TABLE}")
else:
    print(f"   ✅ PASSED — all rows have enrolled_count >= 1")

In [0]:
# ============================================================
# CELL 23 — DQ Check 2: all pct columns must be in [0.0, 1.0]
#           An out-of-range value means the count/enrolled
#           division produced an impossible result
# ============================================================

funnel_df = spark.table(FUNNEL_GOLD_TABLE)

pct_cols = ["started_pct", "q1_pct", "q2_pct", "q3_pct", "completed_pct"]

for col_name in pct_cols:
    invalid_rows = funnel_df.filter(
        (F.col(col_name) < 0.0) | (F.col(col_name) > 1.0)
    )
    invalid_count = invalid_rows.count()

    print(f"── DQ Check 2: {col_name} in [0, 1] ────────────")
    if invalid_count > 0:
        (
            invalid_rows
            .withColumn("dq_check_name", F.lit(f"{col_name}_out_of_range"))
            .withColumn("dq_table",      F.lit(FUNNEL_GOLD_TABLE))
            .withColumn("dq_severity",   F.lit("HIGH"))
            .withColumn("dq_checked_at", F.current_timestamp())
            .withColumn("dq_message",    F.concat_ws(" | ",
                F.lit(f"{col_name} out of [0,1]:"),
                F.col(col_name).cast("string"),
                F.lit("course_id:"), F.col("course_id")
            ))
            .select("dq_check_name", "dq_table", "dq_severity",
                    "dq_checked_at", "dq_message", "course_id")
            .write.format("delta").mode("append")
            .option("mergeSchema", "true")
            .saveAsTable(DQ_AUDIT_TABLE)
        )
        print(f"   ⚠ {invalid_count} rows out of range → {DQ_AUDIT_TABLE}")
    else:
        print(f"   ✅ PASSED")
    print()

In [0]:
# ============================================================
# CELL 24 — DQ Check 3: funnel monotonicity
#           Each stage pct must be >= the next stage pct
#           (you cannot reach 50% without first reaching 25%)
#
#           Violations indicate a progress_pct data error
#           in the upstream enrollments source.
# ============================================================

funnel_df = spark.table(FUNNEL_GOLD_TABLE)

monotonicity_violations = funnel_df.filter(
    (F.col("started_pct") < F.col("q1_pct")) |
    (F.col("q1_pct") < F.col("q2_pct")) |
    (F.col("q2_pct") < F.col("q3_pct")) |
    (F.col("q3_pct") < F.col("completed_pct"))
)

violation_count = monotonicity_violations.count()
print("── DQ Check 3: funnel monotonicity ───────────────")
print(f"   Violations (later stage > earlier stage): {violation_count}")

if violation_count > 0:
    (
        monotonicity_violations
        .withColumn("dq_check_name", F.lit("funnel_not_monotonic"))
        .withColumn("dq_table",      F.lit(FUNNEL_GOLD_TABLE))
        .withColumn("dq_severity",   F.lit("HIGH"))
        .withColumn("dq_checked_at", F.current_timestamp())
        .withColumn("dq_message",    F.concat_ws(" | ",
            F.lit("Funnel not monotonically decreasing for course_id:"),
            F.col("course_id"),
            F.lit("started/q1/q2/q3/completed:"),
            F.col("started_pct").cast("string"),
            F.col("q1_pct").cast("string"),
            F.col("q2_pct").cast("string"),
            F.col("q3_pct").cast("string"),
            F.col("completed_pct").cast("string")
        ))
        .select("dq_check_name", "dq_table", "dq_severity",
                "dq_checked_at", "dq_message", "course_id")
        .write.format("delta").mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(DQ_AUDIT_TABLE)
    )
    print(f"   ⚠ {violation_count} rows flagged → {DQ_AUDIT_TABLE}")
    monotonicity_violations.select(
        "course_id", "started_pct", "q1_pct",
        "q2_pct", "q3_pct", "completed_pct"
    ).show(5, truncate=False)
else:
    print("   ✅ PASSED — funnel is monotonically non-increasing for all courses")

In [0]:
# ============================================================
# CELL 25 — DQ Check 4: biggest_dropoff_stage must be one
#           of the 5 valid transition labels
# ============================================================

funnel_df = spark.table(FUNNEL_GOLD_TABLE)

invalid_stage = funnel_df.filter(
    ~F.col("biggest_dropoff_stage").isin(STAGE_TRANSITIONS)
)

invalid_stage_count = invalid_stage.count()
print("── DQ Check 4: biggest_dropoff_stage valid values ─")
print(f"   Valid values: {STAGE_TRANSITIONS}")
if invalid_stage_count > 0:
    (
        invalid_stage
        .withColumn("dq_check_name", F.lit("invalid_biggest_dropoff_stage"))
        .withColumn("dq_table",      F.lit(FUNNEL_GOLD_TABLE))
        .withColumn("dq_severity",   F.lit("MEDIUM"))
        .withColumn("dq_checked_at", F.current_timestamp())
        .withColumn("dq_message",    F.concat_ws(" | ",
            F.lit("biggest_dropoff_stage has invalid value:"),
            F.col("biggest_dropoff_stage"),
            F.lit("course_id:"), F.col("course_id")
        ))
        .select("dq_check_name", "dq_table", "dq_severity",
                "dq_checked_at", "dq_message", "course_id")
        .write.format("delta").mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(DQ_AUDIT_TABLE)
    )
    print(f"   ⚠ {invalid_stage_count} rows with invalid stage → {DQ_AUDIT_TABLE}")
else:
    print("   ✅ PASSED — all biggest_dropoff_stage values are valid")

In [0]:
%sql
-- ============================================================
-- CELL 26 — SQL verification of gold.course_completion_funnel
-- ============================================================

SELECT
    COUNT(*)                                          AS total_courses,
    SUM(enrolled_count)                               AS total_enrolled_students,
    ROUND(AVG(started_pct), 4)                        AS avg_started_pct,
    ROUND(AVG(q1_pct), 4)                             AS avg_q1_pct,
    ROUND(AVG(q2_pct), 4)                             AS avg_q2_pct,
    ROUND(AVG(q3_pct), 4)                             AS avg_q3_pct,
    ROUND(AVG(completed_pct), 4)                      AS avg_completed_pct,
    ROUND(AVG(cohort_q1_completion), 4)               AS avg_q1_cohort_completion,
    ROUND(AVG(cohort_q2_completion), 4)               AS avg_q2_cohort_completion,
    SUM(CASE WHEN cohort_improvement = 'IMPROVED' THEN 1 ELSE 0 END) AS improved_courses,
    SUM(CASE WHEN cohort_improvement = 'DECLINED' THEN 1 ELSE 0 END) AS declined_courses,
    SUM(CASE WHEN cohort_improvement = 'STABLE'   THEN 1 ELSE 0 END) AS stable_courses
FROM mini_project_grp6.gold.course_completion_funnel;

In [0]:
%sql
-- ============================================================
-- CELL 27 — Which funnel stage loses the most students?
--           Aggregated across all courses — tells the business
--           where to focus curriculum improvement effort.
-- ============================================================

SELECT
    biggest_dropoff_stage,
    COUNT(*)                                 AS courses_with_this_dropoff,
    ROUND(AVG(ABS(min_drop)), 4)             AS avg_absolute_drop,
    ROUND(AVG(enrolled_count), 0)            AS avg_enrolled_per_course
FROM mini_project_grp6.gold.course_completion_funnel
GROUP BY biggest_dropoff_stage
ORDER BY courses_with_this_dropoff DESC;

In [0]:
# ============================================================
# CELL 28 — Task 3.3 completion summary + Full Gold Layer done
# ============================================================

final_df      = spark.table(FUNNEL_GOLD_TABLE)
final_count   = final_df.count()

avg_completed = final_df.agg(
    F.round(F.avg("completed_pct"), 4)
).collect()[0][0]

improved_count = final_df.filter(
    F.col("cohort_improvement") == "IMPROVED"
).count()

worst_stage = final_df.groupBy("biggest_dropoff_stage") \
    .count() \
    .orderBy(F.desc("count")) \
    .first()[0]

print("═" * 62)
print("  TASK 3.3 COMPLETE — Gold Course Completion Funnel")
print("═" * 62)
print()
print(f"  ✅ {FUNNEL_GOLD_TABLE}")
print(f"      Rows (one per course)     : {final_count:,}")
print(f"      Mode                      : OVERWRITE (full recompute)")
print()
print("  Funnel stages computed:")
print("      enrolled → started → 25% → 50% → 75% → completed")
print(f"      Avg completion rate across all courses: {avg_completed:.2%}")
print(f"      Most common dropoff stage             : {worst_stage}")
print()
print("  Cohort analysis:")
print(f"      Q1 cohort : {Q1_START} → {Q1_END}")
print(f"      Q2 cohort : {Q2_START} → {Q2_END}")
print(f"      Courses where Q2 improved over Q1: {improved_count:,}")
print()
print("  DQ checks run:")
print("      Cell 22 — enrolled_count >= 1")
print("      Cell 23 — all pct columns in [0, 1]")
print("      Cell 24 — funnel monotonicity")
print("      Cell 25 — biggest_dropoff_stage valid values")
print()
print("  ─────────────────────────────────────────────────────")
print("  ✅ FULL GOLD LAYER COMPLETE — all 3 tables done")
print()
print("  Gold tables:")
print("      mini_project_grp6.gold.course_performance_daily")
print("      mini_project_grp6.gold.instructor_dashboard")
print("      mini_project_grp6.gold.course_completion_funnel")
print()
print("  ─────────────────────────────────────────────────────")
print("  Next: Task 4.1 → 11_airflow_dag")
print("         edtech_pipeline DAG — hourly Bronze + daily Silver/Gold")
print("═" * 62)